# GDD Data Cleaning

## Purpose
Clean Global Dietary Database country-level dietary intake estimates for the nutrition-transition dashboard.

## Inputs
- Original GDD country-level estimate CSV files in `data/raw/`.
- The accompanying GDD codebook in `data/reference/`.

## Outputs
- A cleaned long-format dietary intake table.
- A country-year table for integration with the other datasets.
- Cleaned outputs will be saved in `data/cleaned/`.

## Important framing
GDD values are modelled dietary intake estimates. Population groups, units, energy adjustment, and uncertainty must be documented when interpreting or comparing the estimates.

Original downloaded files are kept unchanged. All cleaning is performed in this notebook and saved as separate output files.


In [3]:
from pathlib import Path
import pandas as pd

# Works when the notebook runs from the project root or notebooks folder
current = Path.cwd()

if current.name == "notebooks":
    ROOT = current.parent
elif (current / "notebooks").is_dir():
    ROOT = current
else:
    raise FileNotFoundError(
        "Open the project root or run this notebook from notebooks/."
    )

RAW = ROOT / "data" / "raw" / "gdd"
CLEANED = ROOT / "data" / "cleaned"

CLEANED.mkdir(parents=True, exist_ok=True)

print("Raw data folder:", RAW)
print("Output folder:", CLEANED)

Raw data folder: /Users/syakira/Documents/GitHub/vabi-YouAreWhatYouEat/data/raw/gdd
Output folder: /Users/syakira/Documents/GitHub/vabi-YouAreWhatYouEat/data/cleaned


In [4]:
files = sorted(RAW.rglob("*.csv"))

print("Number of CSV files:", len(files))

for file in files[:30]:
    print(file.relative_to(RAW))

if not files:
    raise FileNotFoundError(
        "No CSV files found. Extract the GDD downloads into data/raw/gdd/."
    )

Number of CSV files: 47
v01_cnty.csv
v02_cnty.csv
v03_cnty.csv
v04_cnty.csv
v05_cnty.csv
v06_cnty.csv
v07_cnty.csv
v08_cnty.csv
v09_cnty.csv
v10_cnty.csv
v11_cnty.csv
v12_cnty.csv
v13_cnty.csv
v14_cnty.csv
v15_cnty.csv
v16_cnty.csv
v17_cnty.csv
v18_cnty.csv
v22_cnty.csv
v23_cnty.csv
v27_cnty.csv
v28_cnty.csv
v29_cnty.csv
v30_cnty.csv
v31_cnty.csv
v33_cnty.csv
v34_cnty.csv
v35_cnty.csv
v36_cnty.csv
v37_cnty.csv


In [5]:
sample_file = files[0]
sample = pd.read_csv(sample_file, low_memory=False)

print("File:", sample_file.name)
print("Rows and columns:", sample.shape)
print("\nColumn names:")
print(sample.columns.tolist())

display(sample.head())
sample.info()

File: v01_cnty.csv
Rows and columns: (1072260, 16)

Column names:
['superregion2', 'iso3', 'age', 'female', 'urban', 'edu', 'year', 'varnum', 'v01_type', 'v01_type_desc', 'median', 'upperci_95', 'lowerci_95', 'serving', 's_lowerci_95', 's_upperci_95']


,superregion2,iso3,age,female,urban,edu,year,varnum,v01_type,v01_type_desc,median,upperci_95,lowerci_95,serving,s_lowerci_95,s_upperci_95
0,SAARC,AFG,0.5,0,0,1,1990,1,1,1: Foods & Beverages,10.198521,21.203149,4.847648,0.208133,0.098932,0.432717
1,SAARC,AFG,0.5,0,0,2,1990,1,1,1: Foods & Beverages,14.882476,31.914361,7.066668,0.303724,0.144218,0.651313
2,SAARC,AFG,0.5,0,0,3,1990,1,1,1: Foods & Beverages,19.841635,41.671546,9.462270,0.404931,0.193108,0.850440
3,SAARC,AFG,0.5,0,0,999,1990,1,1,1: Foods & Beverages,11.303815,21.286601,6.259829,0.230690,0.127752,0.434420
4,SAARC,AFG,0.5,0,1,1,1990,1,1,1: Foods & Beverages,14.171020,29.690704,6.785812,0.289204,0.138486,0.605933


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1072260 entries, 0 to 1072259
Data columns (total 16 columns):
 #   Column         Non-Null Count    Dtype  
---  ------         --------------    -----  
 0   superregion2   1072260 non-null  object 
 1   iso3           1072260 non-null  object 
 2   age            1072260 non-null  float64
 3   female         1072260 non-null  int64  
 4   urban          1072260 non-null  int64  
 5   edu            1072260 non-null  int64  
 6   year           1072260 non-null  int64  
 7   varnum         1072260 non-null  int64  
 8   v01_type       1072260 non-null  int64  
 9   v01_type_desc  1072260 non-null  object 
 10  median         1068312 non-null  float64
 11  upperci_95     1068312 non-null  float64
 12  lowerci_95     1068312 non-null  float64
 13  serving        1068312 non-null  float64
 14  s_lowerci_95   1068312 non-null  float64
 15  s_upperci_95   1068312 non-null  float64
dtypes: float64(7), int64(6), object(3)
memory usage: 130.9

In [6]:
display(
    sample.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_rows")
)

,missing_rows
median,3948
upperci_95,3948
lowerci_95,3948
serving,3948
s_lowerci_95,3948
s_upperci_95,3948
superregion2,0
iso3,0
age,0
female,0


## Inspect population groups and coverage

Check the demographic codes, years, countries, and dietary factor
before choosing a consistent population for the dashboard.

In [7]:
# Show the available demographic codes and years
for column in ["age", "female", "urban", "edu", "year", "varnum"]:
    print(f"{column}:")
    print(sorted(sample[column].dropna().unique()))
    print()

print("Number of countries:", sample["iso3"].nunique())

age:
[np.float64(0.5), np.float64(1.5), np.float64(3.5), np.float64(7.5), np.float64(12.5), np.float64(17.5), np.float64(22.5), np.float64(27.5), np.float64(32.5), np.float64(37.5), np.float64(42.5), np.float64(47.5), np.float64(52.5), np.float64(57.5), np.float64(62.5), np.float64(67.5), np.float64(72.5), np.float64(77.5), np.float64(82.5), np.float64(87.5), np.float64(92.5), np.float64(97.5), np.float64(999.0)]

female:
[np.int64(0), np.int64(1), np.int64(999)]

urban:
[np.int64(0), np.int64(1), np.int64(999)]

edu:
[np.int64(1), np.int64(2), np.int64(3), np.int64(999)]

year:
[np.int64(1990), np.int64(1995), np.int64(2000), np.int64(2005), np.int64(2010), np.int64(2015), np.int64(2018)]

varnum:
[np.int64(1)]

Number of countries: 185


In [8]:
# Inspect potential overall groups without removing any data
overall = sample.loc[
    sample["female"].eq(999)
    & sample["urban"].eq(999)
    & sample["edu"].eq(999)
]

print("Rows with female=999, urban=999, edu=999:", len(overall))
print("Available age codes in these rows:")
print(sorted(overall["age"].dropna().unique()))

display(overall.head(10))

Rows with female=999, urban=999, edu=999: 29785
Available age codes in these rows:
[np.float64(0.5), np.float64(1.5), np.float64(3.5), np.float64(7.5), np.float64(12.5), np.float64(17.5), np.float64(22.5), np.float64(27.5), np.float64(32.5), np.float64(37.5), np.float64(42.5), np.float64(47.5), np.float64(52.5), np.float64(57.5), np.float64(62.5), np.float64(67.5), np.float64(72.5), np.float64(77.5), np.float64(82.5), np.float64(87.5), np.float64(92.5), np.float64(97.5), np.float64(999.0)]


,superregion2,iso3,age,female,urban,edu,year,varnum,v01_type,v01_type_desc,median,upperci_95,lowerci_95,serving,s_lowerci_95,s_upperci_95
35,SAARC,AFG,0.5,999,999,999,1990,1,1,1: Foods & Beverages,12.939732,19.735003,8.597366,0.264076,0.175456,0.402755
71,SAARC,AFG,1.5,999,999,999,1990,1,1,1: Foods & Beverages,26.249086,39.709005,17.912155,0.349988,0.238829,0.529453
107,SAARC,AFG,3.5,999,999,999,1990,1,1,1: Foods & Beverages,45.437017,69.014567,31.051544,0.568674,0.388630,0.863762
143,SAARC,AFG,7.5,999,999,999,1990,1,1,1: Foods & Beverages,73.571181,110.852589,50.092301,0.735712,0.500923,1.108526
179,SAARC,AFG,12.5,999,999,999,1990,1,1,1: Foods & Beverages,93.177377,139.616393,64.289241,0.931774,0.642892,1.396164
215,SAARC,AFG,17.5,999,999,999,1990,1,1,1: Foods & Beverages,98.048766,145.639023,66.414548,0.980488,0.664145,1.456390
251,SAARC,AFG,22.5,999,999,999,1990,1,1,1: Foods & Beverages,94.615732,142.613892,64.443235,0.946157,0.644432,1.426139
287,SAARC,AFG,27.5,999,999,999,1990,1,1,1: Foods & Beverages,88.411161,131.873334,59.594313,0.884112,0.595943,1.318733
323,SAARC,AFG,32.5,999,999,999,1990,1,1,1: Foods & Beverages,83.169545,123.452602,56.611932,0.831695,0.566119,1.234526
359,SAARC,AFG,37.5,999,999,999,1990,1,1,1: Foods & Beverages,78.895885,119.858251,53.896697,0.788959,0.538967,1.198583


In [9]:
missing = sample.loc[sample["median"].isna()]

print("Rows with missing intake estimates:", len(missing))

display(
    missing.groupby("age", dropna=False)
    .size()
    .reset_index(name="missing_rows")
)

display(
    missing[
        ["iso3", "year", "age", "female", "urban", "edu", "median"]
    ].head(20)
)

Rows with missing intake estimates: 3948


,age,missing_rows
0,0.5,132
1,1.5,132
2,3.5,132
3,7.5,132
4,12.5,132
5,17.5,132
6,22.5,132
7,27.5,132
8,32.5,132
9,37.5,132


,iso3,year,age,female,urban,edu,median
35544,ATG,1990,97.5,1,0,1,NaN
35545,ATG,1990,97.5,1,0,2,NaN
35546,ATG,1990,97.5,1,0,3,NaN
35547,ATG,1990,97.5,1,0,999,NaN
35548,ATG,1990,97.5,1,1,1,NaN
35549,ATG,1990,97.5,1,1,2,NaN
35550,ATG,1990,97.5,1,1,3,NaN
35551,ATG,1990,97.5,1,1,999,NaN
35552,ATG,1990,97.5,1,999,1,NaN
35553,ATG,1990,97.5,1,999,2,NaN


In [10]:
for file in files:
    print(file.relative_to(RAW))

v01_cnty.csv
v02_cnty.csv
v03_cnty.csv
v04_cnty.csv
v05_cnty.csv
v06_cnty.csv
v07_cnty.csv
v08_cnty.csv
v09_cnty.csv
v10_cnty.csv
v11_cnty.csv
v12_cnty.csv
v13_cnty.csv
v14_cnty.csv
v15_cnty.csv
v16_cnty.csv
v17_cnty.csv
v18_cnty.csv
v22_cnty.csv
v23_cnty.csv
v27_cnty.csv
v28_cnty.csv
v29_cnty.csv
v30_cnty.csv
v31_cnty.csv
v33_cnty.csv
v34_cnty.csv
v35_cnty.csv
v36_cnty.csv
v37_cnty.csv
v38_cnty.csv
v39_cnty.csv
v40_cnty.csv
v41_cnty.csv
v42_cnty.csv
v43_cnty.csv
v45_cnty.csv
v46_cnty.csv
v47_cnty.csv
v48_cnty.csv
v49_cnty.csv
v50_cnty.csv
v51_cnty.csv
v52_cnty.csv
v53_cnty.csv
v54_cnty.csv
v57_cnty.csv


## Select national overall estimates

Keep the GDD-provided estimates for all ages, both sexes,
all residences, and all education levels.

These estimates describe the overall population, not adults only.
Do not average demographic subgroup rows together.

In [11]:
national = sample.loc[
    sample["age"].eq(999)
    & sample["female"].eq(999)
    & sample["urban"].eq(999)
    & sample["edu"].eq(999)
].copy()

print("Rows before filtering:", len(sample))
print("Rows after filtering:", len(national))
print("Countries:", national["iso3"].nunique())
print("Years:", sorted(national["year"].unique()))

display(national.head())

Rows before filtering: 1072260
Rows after filtering: 1295
Countries: 185
Years: [np.int64(1990), np.int64(1995), np.int64(2000), np.int64(2005), np.int64(2010), np.int64(2015), np.int64(2018)]


,superregion2,iso3,age,female,urban,edu,year,varnum,v01_type,v01_type_desc,median,upperci_95,lowerci_95,serving,s_lowerci_95,s_upperci_95
827,SAARC,AFG,999.0,999,999,999,1990,1,1,1: Foods & Beverages,76.640482,100.331036,59.471595,0.809298,0.628000,1.059462
1655,SAARC,AFG,999.0,999,999,999,1995,1,1,1: Foods & Beverages,60.214490,77.703949,46.476762,0.635778,0.490727,0.820441
2483,SAARC,AFG,999.0,999,999,999,2000,1,1,1: Foods & Beverages,53.654165,70.045729,41.750782,0.567589,0.441667,0.740989
3311,SAARC,AFG,999.0,999,999,999,2005,1,1,1: Foods & Beverages,49.164932,64.018664,38.134218,0.518617,0.402260,0.675302
4139,SAARC,AFG,999.0,999,999,999,2010,1,1,1: Foods & Beverages,61.053277,79.578400,47.148061,0.642396,0.496087,0.837315


In [12]:
national = national.rename(columns={
    "median": "intake_estimate",
    "lowerci_95": "intake_lower_95",
    "upperci_95": "intake_upper_95",
    "serving": "servings_per_day",
    "s_lowerci_95": "servings_lower_95",
    "s_upperci_95": "servings_upper_95"
})

# This sample is v01, which the codebook identifies as Fruits.
assert national["varnum"].eq(1).all()

national["dietary_factor"] = "Fruits"
national["unit"] = "g/day"
national["population"] = "All ages; both sexes; all residences and education levels"

In [15]:
columns_to_keep = [
    "iso3",
    "superregion2",
    "year",
    "varnum",
    "dietary_factor",
    "unit",
    "population",
    "age",
    "female",
    "urban",
    "edu",
    "intake_estimate",
    "intake_lower_95",
    "intake_upper_95",
    "servings_per_day",
    "servings_lower_95",
    "servings_upper_95"
]

national = national[columns_to_keep].copy()

national["iso3"] = (
    national["iso3"]
    .astype("string")
    .str.strip()
    .str.upper()
)

national = national.sort_values(
    ["iso3", "year"]
).reset_index(drop=True)

display(national.head())

,iso3,superregion2,year,varnum,dietary_factor,unit,population,age,female,urban,edu,intake_estimate,intake_lower_95,intake_upper_95,servings_per_day,servings_lower_95,servings_upper_95
0,AFG,SAARC,1990,1,Fruits,g/day,All ages; both sexes; all residences and educa...,999.0,999,999,999,76.640482,59.471595,100.331036,0.809298,0.628000,1.059462
1,AFG,SAARC,1995,1,Fruits,g/day,All ages; both sexes; all residences and educa...,999.0,999,999,999,60.214490,46.476762,77.703949,0.635778,0.490727,0.820441
2,AFG,SAARC,2000,1,Fruits,g/day,All ages; both sexes; all residences and educa...,999.0,999,999,999,53.654165,41.750782,70.045729,0.567589,0.441667,0.740989
3,AFG,SAARC,2005,1,Fruits,g/day,All ages; both sexes; all residences and educa...,999.0,999,999,999,49.164932,38.134218,64.018664,0.518617,0.402260,0.675302
4,AFG,SAARC,2010,1,Fruits,g/day,All ages; both sexes; all residences and educa...,999.0,999,999,999,61.053277,47.148061,79.578400,0.642396,0.496087,0.837315


In [13]:
# Check repeated country-year records
duplicate_rows = national.loc[
    national.duplicated(["iso3", "year"], keep=False)
]

print("Rows with duplicate country-year keys:", len(duplicate_rows))
display(duplicate_rows)

# Check missing estimates
print(
    "Missing national estimates:",
    national["intake_estimate"].isna().sum()
)

# Check negative values and inconsistent uncertainty bounds
negative = national["intake_estimate"].lt(0)

invalid_bounds = (
    national["intake_lower_95"].gt(national["intake_estimate"])
    | national["intake_estimate"].gt(national["intake_upper_95"])
)

print("Negative estimates:", negative.sum())
print("Inconsistent uncertainty bounds:", invalid_bounds.sum())

display(national.loc[negative | invalid_bounds])

# Check coverage by year
display(
    national.groupby("year").agg(
        countries=("iso3", "nunique"),
        available_estimates=("intake_estimate", "count")
    )
)

Rows with duplicate country-year keys: 0


,superregion2,iso3,age,female,urban,edu,year,varnum,v01_type,v01_type_desc,intake_estimate,intake_upper_95,intake_lower_95,servings_per_day,servings_lower_95,servings_upper_95,dietary_factor,unit,population


Missing national estimates: 0
Negative estimates: 0
Inconsistent uncertainty bounds: 0


,superregion2,iso3,age,female,urban,edu,year,varnum,v01_type,v01_type_desc,intake_estimate,intake_upper_95,intake_lower_95,servings_per_day,servings_lower_95,servings_upper_95,dietary_factor,unit,population


,countries,available_estimates
year,,
1990,185,185
1995,185,185
2000,185,185
2005,185,185
2010,185,185
2015,185,185
2018,185,185


## Clean all GDD country-level files

Read each file in chunks and retain only the national overall population.
Attach dietary-factor names and units from the GDD codebook.
Preserve missing estimates and uncertainty intervals.

In [18]:
%pip install openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]1/2 [openpyxl]

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [14]:
REF = ROOT / "data" / "reference"

codebooks = sorted(REF.glob("GDD*Codebook*.xlsx"))

if len(codebooks) != 1:
    raise ValueError(
        f"Expected one GDD codebook in {REF}, found {len(codebooks)}. "
        "Set codebook_path to the correct file explicitly."
    )

codebook_path = codebooks[0]
print("Using:", codebook_path.name)

# Dietary-factor codes and names
factors = pd.read_excel(
    codebook_path,
    sheet_name="File nomenclature",
    header=1,
    usecols="A:B"
)

factors.columns = ["factor_code", "dietary_factor"]

factors = factors.loc[
    factors["factor_code"].astype("string").str.fullmatch(r"v\d+", na=False)
].copy()

factors["varnum"] = (
    factors["factor_code"].str.removeprefix("v").astype(int)
)

# Dietary-factor units
units = pd.read_excel(
    codebook_path,
    sheet_name="Dietary intake data",
    header=1,
    usecols="A:B"
)

units.columns = ["unit_label", "unit"]

# Three labels differ slightly between codebook sheets
label_aliases = {
    "Monounsaturated fatty acids": "Monounsaturated fat",
    "Total omega-6 fat": "Total omega-6 fatty acids",
    "Plant omega-3 fat": "Plant omega-3 (n-3) fat",
    "Seafood omega-3 fat": "Seafood omega-3 (n-3) fat",
    "Vitamin A w/ supplements": "Vitamin A with supplements"
}

factors["unit_label"] = factors["dietary_factor"].replace(label_aliases)

factor_reference = factors.merge(
    units,
    on="unit_label",
    how="left",
    validate="many_to_one"
)

unmatched = factor_reference.loc[factor_reference["unit"].isna()]
display(unmatched)

assert unmatched.empty, "Some dietary factors have no matched unit."

display(factor_reference[[
    "factor_code", "varnum", "dietary_factor", "unit"
]])

Using: GDD 2018 Codebook_Jan 10 2022.xlsx


,factor_code,dietary_factor,varnum,unit_label,unit


,factor_code,varnum,dietary_factor,unit
0,v01,1,Fruits,grams per day
1,v02,2,Non-starchy vegetables,grams per day
2,v03,3,Potatoes,grams per day
3,v04,4,Other starchy vegetables,grams per day
4,v05,5,Beans and legumes,grams per day
5,v06,6,Nuts and seeds,grams per day
6,v07,7,Refined grains,grams per day
7,v08,8,Whole grains,grams per day
8,v09,9,Total processed meats,grams per day
9,v10,10,Unprocessed red meats,grams per day


In [15]:
gdd_files = sorted(RAW.glob("v*_cnty.csv"))

expected_codes = set(factor_reference["factor_code"])
actual_codes = {file.stem.split("_")[0] for file in gdd_files}

print("Files found:", len(gdd_files))
print("Missing files:", sorted(expected_codes - actual_codes))
print("Unexpected files:", sorted(actual_codes - expected_codes))

assert actual_codes == expected_codes, (
    "Resolve the file differences before continuing."
)

Files found: 47
Missing files: []
Unexpected files: []


In [18]:
demographics = ["age", "female", "urban", "edu"]

required_columns = [
    "iso3", "superregion2", "year", "varnum",
    *demographics,
    "median", "lowerci_95", "upperci_95"
]

rename_columns = {
    "median": "intake_estimate",
    "lowerci_95": "intake_lower_95",
    "upperci_95": "intake_upper_95",
    "serving": "servings_per_day",
    "s_lowerci_95": "servings_lower_95",
    "s_upperci_95": "servings_upper_95"
}

parts = []
audit_rows = []

for file in gdd_files:
    factor_code = file.stem.split("_")[0]
    expected_varnum = int(factor_code[1:])

    selected_parts = []
    rows_read = 0

    for chunk in pd.read_csv(
        file,
        chunksize=100_000,
        low_memory=False
    ):
        # Recover the factor number from the filename if absent
        if "varnum" not in chunk.columns:
            chunk["varnum"] = expected_varnum
        
        missing_columns = set(required_columns) - set(chunk.columns)

        if missing_columns:
            raise ValueError(
                f"{file.name}: missing columns {missing_columns}"
            )

        rows_read += len(chunk)

        if not chunk["varnum"].eq(expected_varnum).all():
            raise ValueError(f"Unexpected varnum in {file.name}")

        mask = chunk[demographics].eq(999).all(axis=1)

        selected = chunk.loc[mask].copy()

        # Serving fields may not exist for every dietary factor.
        available = required_columns + [
            column for column in
            ["serving", "s_lowerci_95", "s_upperci_95"]
            if column in chunk.columns
        ]

        selected_parts.append(selected[available])

    selected = pd.concat(selected_parts, ignore_index=True)
    selected = selected.rename(columns=rename_columns)

    if selected.empty:
        raise ValueError(f"No national overall rows in {file.name}")

    selected["source_file"] = file.name
    parts.append(selected)

    audit_rows.append({
        "source_file": file.name,
        "rows_read": rows_read,
        "rows_retained": len(selected),
        "missing_estimates": selected["intake_estimate"].isna().sum()
    })

    print(f"{file.name}: retained {len(selected):,} rows")

gdd = pd.concat(parts, ignore_index=True)

gdd = gdd.merge(
    factor_reference[[
        "varnum", "factor_code", "dietary_factor", "unit"
    ]],
    on="varnum",
    how="left",
    validate="many_to_one"
)

gdd["iso3"] = (
    gdd["iso3"].astype("string").str.strip().str.upper()
)

gdd["population"] = (
    "All ages; both sexes; all residences and education levels"
)

gdd = gdd.sort_values(
    ["iso3", "year", "varnum"]
).reset_index(drop=True)

audit = pd.DataFrame(audit_rows)

display(audit)
print("Combined shape:", gdd.shape)

v01_cnty.csv: retained 1,295 rows
v02_cnty.csv: retained 1,295 rows
v03_cnty.csv: retained 1,295 rows
v04_cnty.csv: retained 1,295 rows
v05_cnty.csv: retained 1,295 rows
v06_cnty.csv: retained 1,295 rows
v07_cnty.csv: retained 1,295 rows
v08_cnty.csv: retained 1,295 rows
v09_cnty.csv: retained 1,295 rows
v10_cnty.csv: retained 1,480 rows
v11_cnty.csv: retained 1,295 rows
v12_cnty.csv: retained 1,295 rows
v13_cnty.csv: retained 1,295 rows
v14_cnty.csv: retained 1,295 rows
v15_cnty.csv: retained 1,295 rows
v16_cnty.csv: retained 1,295 rows
v17_cnty.csv: retained 1,295 rows
v18_cnty.csv: retained 1,295 rows
v22_cnty.csv: retained 1,295 rows
v23_cnty.csv: retained 1,295 rows
v27_cnty.csv: retained 1,295 rows
v28_cnty.csv: retained 1,295 rows
v29_cnty.csv: retained 1,295 rows
v30_cnty.csv: retained 1,295 rows
v31_cnty.csv: retained 1,295 rows
v33_cnty.csv: retained 1,480 rows
v34_cnty.csv: retained 1,295 rows
v35_cnty.csv: retained 1,480 rows
v36_cnty.csv: retained 1,295 rows
v37_cnty.csv: 

,source_file,rows_read,rows_retained,missing_estimates
0,v01_cnty.csv,1072260,1295,0
1,v02_cnty.csv,1072260,1295,0
2,v03_cnty.csv,1072260,1295,0
3,v04_cnty.csv,1072260,1295,0
4,v05_cnty.csv,1072260,1295,0
5,v06_cnty.csv,1072260,1295,0
6,v07_cnty.csv,1072260,1295,0
7,v08_cnty.csv,1072260,1295,0
8,v09_cnty.csv,1072260,1295,0
9,v10_cnty.csv,834720,1480,0


Combined shape: (61605, 19)


In [19]:
key = ["iso3", "year", "varnum"]

duplicates = gdd.duplicated(key, keep=False)

negative = gdd["intake_estimate"].lt(0)

invalid_bounds = (
    gdd["intake_lower_95"].gt(gdd["intake_estimate"])
    | gdd["intake_estimate"].gt(gdd["intake_upper_95"])
)

print("Dietary factors:", gdd["varnum"].nunique())
print("Countries:", gdd["iso3"].nunique())
print("Years:", sorted(gdd["year"].unique()))
print("Duplicate-key rows:", duplicates.sum())
print("Missing estimates:", gdd["intake_estimate"].isna().sum())
print("Negative estimates:", negative.sum())
print("Inconsistent bounds:", invalid_bounds.sum())

display(gdd.loc[duplicates | negative | invalid_bounds])

display(
    gdd.groupby(["factor_code", "dietary_factor"]).agg(
        rows=("year", "size"),
        countries=("iso3", "nunique"),
        years=("year", "nunique"),
        available_estimates=("intake_estimate", "count")
    )
)

assert not gdd[["iso3", "year", "varnum"]].isna().any().any()
assert gdd["unit"].notna().all()
assert not duplicates.any(), "Investigate repeated keys."
assert not negative.any(), "Investigate negative estimates."
assert not invalid_bounds.any(), "Investigate uncertainty bounds."

Dietary factors: 47
Countries: 185
Years: [np.int64(1990), np.int64(1995), np.int64(2000), np.int64(2005), np.int64(2010), np.int64(2015), np.int64(2018), np.int64(2020)]
Duplicate-key rows: 0
Missing estimates: 0
Negative estimates: 0
Inconsistent bounds: 0


,iso3,superregion2,year,varnum,age,female,urban,edu,intake_estimate,intake_lower_95,intake_upper_95,servings_per_day,servings_lower_95,servings_upper_95,source_file,factor_code,dietary_factor,unit,population


,,rows,countries,years,available_estimates
factor_code,dietary_factor,,,,
v01,Fruits,1295,185,7,1295
v02,Non-starchy vegetables,1295,185,7,1295
v03,Potatoes,1295,185,7,1295
v04,Other starchy vegetables,1295,185,7,1295
v05,Beans and legumes,1295,185,7,1295
v06,Nuts and seeds,1295,185,7,1295
v07,Refined grains,1295,185,7,1295
v08,Whole grains,1295,185,7,1295
v09,Total processed meats,1295,185,7,1295


In [21]:
# Long table: estimates, units, population, and uncertainty
gdd.to_csv(
    CLEANED / "gdd_cleaned_long.csv",
    index=False
)

# Identify years containing all dietary factors for every country
coverage = (
    gdd.groupby(["iso3", "year"])["factor_code"]
    .nunique()
    .unstack("iso3")
)

shared_years = coverage.index[
    coverage.eq(len(factor_reference)).all(axis=1)
].tolist()

print("Years with complete factor coverage:", shared_years)

# Create the main merge table using those years
gdd_shared = gdd.loc[gdd["year"].isin(shared_years)]

wide = gdd_shared.pivot(
    index=["iso3", "year"],
    columns="factor_code",
    values="intake_estimate"
).reset_index()

wide.columns.name = None

wide.to_csv(
    CLEANED / "gdd_country_year.csv",
    index=False
)

# Reference explains each wide-table factor column and its unit
factor_reference[[
    "factor_code", "varnum", "dietary_factor", "unit"
]].to_csv(
    CLEANED / "gdd_factor_reference.csv",
    index=False
)

audit.to_csv(
    CLEANED / "gdd_cleaning_audit.csv",
    index=False
)

print("Saved all four outputs to:", CLEANED)
print("Long table:", gdd.shape)
print("Country-year table:", wide.shape)

Years with complete factor coverage: [1990, 1995, 2000, 2005, 2010, 2015, 2018]
Saved all four outputs to: /Users/syakira/Documents/GitHub/vabi-YouAreWhatYouEat/data/cleaned
Long table: (61605, 19)
Country-year table: (1295, 49)


## GDD Cleaned Output Files

| File | What each row represents | Contribution to the project |
|---|---|---|
| `gdd_cleaned_long.csv` | One country, one year, and one dietary factor. Contains 61,605 rows, including intake estimates, units, uncertainty intervals, and source filenames. | Main dietary dataset for Tableau: supports country maps, dietary trends, and selecting different foods or nutrients. Its uncertainty intervals can support tooltips or error bars. |
| `gdd_country_year.csv` | One country and year, with 47 dietary factors in separate columns. Contains 1,295 rows covering 185 countries and seven shared years, from 1990 to 2018. | Main file for merging with WHO: join using `iso3` and `year` to explore relationships between dietary intake and health outcomes. |
| `gdd_factor_reference.csv` | One dietary factor, showing its code, name, and measurement unit. Contains 47 rows. | Data dictionary: explains that `v01` means fruits, for example. Helps teammates label charts and interpret values correctly. |
| `gdd_cleaning_audit.csv` | One original CSV, recording rows read, rows retained, and missing intake estimates after filtering. Contains 47 rows. | Cleaning documentation: demonstrates what the notebook processed and supports the methodology section of the report. |